# Part 1 : Snowflake 基礎

## このノートブックでやること
1. 仮想ウェアハウスの作成と設定
2. ステージからのデータ取り込み
3. 【デモ】Snowflake Marketplace からの外部データ活用

## 使用データ
GlacierStyle - 架空 EC サイトのサンプルデータセット


In [ ]:
-- コンテキスト設定
USE ROLE ACCOUNTADMIN;
USE DATABASE GLACIERSTYLE_DB;
USE SCHEMA EC_ANALYTICS_SCHEMA;


## 1. 仮想ウェアハウスの作成と設定

仮想ウェアハウスは、クエリやデータ処理を実行するための**コンピューティングリソース**です。
Snowflake ではストレージとコンピューティングが分離されているため、ウェアハウスを自由にスケールアップ/ダウンできます。

![WH Overview](images/part1/01_warehouse_overview.png)

### 主なパラメータ

![WH Params](images/part1/02_warehouse_params.png)

| パラメータ | 説明 | デフォルト |
|---|---|---|
| `WAREHOUSE_SIZE` | コンピューティングサイズ（X-Small 〜 X-Large 等） | XSmall |
| `AUTO_SUSPEND` | 非アクティブ後に自動停止するまでの秒数 | 600秒 |
| `AUTO_RESUME` | クエリ送信時に自動で再開するか | TRUE |
| `INITIALLY_SUSPENDED` | 作成直後に停止状態で開始するか | TRUE |


In [ ]:
-- 既存のウェアハウスを確認
SHOW WAREHOUSES;


In [ ]:
-- ウェアハウスを作成
-- INITIALLY_SUSPENDED = true → 作成直後は停止状態
-- AUTO_RESUME = false        → 手動で再開が必要（次のステップで体験）
CREATE OR REPLACE WAREHOUSE my_wh
    WAREHOUSE_TYPE      = 'standard'
    WAREHOUSE_SIZE      = 'xsmall'
    AUTO_SUSPEND        = 60
    INITIALLY_SUSPENDED = true
    AUTO_RESUME         = false;


In [ ]:
-- このワークシートで使用するウェアハウスを指定
USE WAREHOUSE my_wh;


In [ ]:
-- ウェアハウスが停止中のままクエリを実行してみる
-- → エラーが返ってくる（停止中は実行できない）
SELECT * FROM dim_customers;


エラーが出ましたね。ウェアハウスが停止中のため、クエリを実行できません。

`ALTER WAREHOUSE ... RESUME` で手動再開、または `AUTO_RESUME = TRUE` に設定すればクエリ送信時に自動で再開されます。


In [ ]:
-- ウェアハウスを手動で再開
ALTER WAREHOUSE my_wh RESUME;


In [ ]:
-- 次回から自動再開するよう設定
ALTER WAREHOUSE my_wh SET AUTO_RESUME = TRUE;


In [ ]:
-- 再開後はクエリが実行できる
SELECT * FROM dim_customers;


### スケールアップ / スケールダウン

SQL コマンド 1 行でウェアハウスサイズを即座に変更できます。
重いクエリを実行するときだけスケールアップして、終わったらスケールダウン、という使い方が基本です。

> **ポイント:** サイズを 1 段上げるとコンピューティングリソースが 2 倍になります。XSmall → Large で 8 倍。


In [ ]:
-- スケールアップ: XSmall → XLarge
ALTER WAREHOUSE my_wh SET WAREHOUSE_SIZE = 'XLarge';


In [ ]:
-- XLarge で集計クエリを実行
SELECT
    c.membership_tier,
    COUNT(DISTINCT o.order_id) AS order_count,
    SUM(o.total_amount)        AS total_sales
FROM fact_orders o
JOIN dim_customers c ON o.customer_id = c.customer_id
GROUP BY c.membership_tier
ORDER BY total_sales DESC;


In [ ]:
-- 作業が終わったらスケールダウン
ALTER WAREHOUSE my_wh SET WAREHOUSE_SIZE = 'XSmall';


---
## 2. ステージからのデータ取り込み

Snowflake の**ステージ**は、データファイルが置かれた場所を参照するオブジェクトです。
`COPY INTO` コマンドを使ってステージからテーブルへデータをロードします。

### ステージの種類

![Stage Types](images/part1/02_external_stage_1.png)

| 種類 | 説明 |
|---|---|
| **内部ステージ** | Snowflake 管理のストレージ。Snowsight からファイルをアップロードできる |
| **外部ステージ** | S3 / Azure Blob / GCS などクラウドストレージを参照 |

今回は **内部ステージ（DATA_STAGE）** を使います。setup.sql で作成済みで、参加者が data/ フォルダのファイルをアップロードしています。

```
ローカルの data/ フォルダ
    ↓  Snowsight でアップロード
内部ステージ (@DATA_STAGE)
    ↓  COPY INTO
テーブル (dim_customers 等)
```

![COPY INTO Flow](images/part1/03_external_stage_2.png)


In [ ]:
-- DATA_STAGE の確認（setup.sql で作成済み）
-- ※ ステージ自体は setup.sql で作成してあります
DESCRIBE STAGE DATA_STAGE;


In [ ]:
-- ステージ内のファイル一覧を確認（アップロード済みのファイルが表示される）
LIST @DATA_STAGE;


In [ ]:
-- ロード先テーブルを作成
-- ※ setup.sql で既に作成・データロード済みですが、ここでは手順を体験します
CREATE OR REPLACE TABLE dim_customers (
    customer_id       VARCHAR PRIMARY KEY,
    email             VARCHAR,
    phone             VARCHAR,
    last_name         VARCHAR,
    first_name        VARCHAR,
    gender            VARCHAR,
    birth_date        DATE,
    postal_code       VARCHAR,
    prefecture        VARCHAR,
    city              VARCHAR,
    address           VARCHAR,
    registration_date DATE,
    membership_tier   VARCHAR,
    total_orders      INTEGER,
    total_spent       DECIMAL(12,2),
    last_order_date   DATE,
    email_opt_in      BOOLEAN,
    app_installed     BOOLEAN
);


In [ ]:
-- ステージからテーブルへデータをロード
COPY INTO dim_customers
FROM @DATA_STAGE/customers.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"');


In [ ]:
-- ロード結果を確認
SELECT * FROM dim_customers LIMIT 20;


---
## 3. 【デモ】Snowflake Marketplace からの外部データ活用

> **このセクションは講師デモです。**

Snowflake Marketplace では、外部企業が提供するデータを**ゼロコピー**で即座に利用できます。

### デモ内容
- Marketplace から **Weather Source（気象データ）** を取得
- 注文データと気象データを JOIN して「天気と売上の関係」を分析

### ゼロコピーとは
- データを自分のアカウントにコピーしない
- 提供元のデータを直接参照するため、常に最新データを利用可能
- ストレージコストは発生しない

> **強み:** 契約してすぐ使える、メンテナンス不要、常に最新


---
## クリーンアップ

作成したオブジェクトを削除する場合は以下を実行してください。


In [ ]:
USE ROLE ACCOUNTADMIN;
DROP WAREHOUSE IF EXISTS my_wh;
-- ※ dim_customers は Part 2 でも使うため削除しない
